# Lab 01 — Konsep Dasar Apache Spark
**Arsitektur, partition, lazy evaluation, DAG, job/stage/task, dan Spark UI**

Notebook ini dirancang untuk environment pelatihan: Spark 3.5.7 standalone, Jupyter, 2 worker, masing-masing 2 core / 2 GB.  
**Prinsip lab:** prediksi → jalankan → amati → jelaskan. Jangan hanya mengejar output akhir.

## Capaian praktik
- Mengidentifikasi Driver, Master/Cluster Manager, Worker, Executor, Partition, Task, Stage, dan Job.
- Membuktikan lazy evaluation dan membedakan transformation vs action.
- Menghubungkan `explain()` dengan Jobs/Stages/Tasks di Spark UI.
- Mengamati pengaruh jumlah partition terhadap paralelisme.

In [ ]:
if "spark" in globals():
    spark.stop()

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T
import time, json, os, shutil

spark = (
    SparkSession.builder
    .appName("spark-training-lab-01")
    .master("spark://spark-master:7077")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark version :", spark.version)
print("Master        :", spark.sparkContext.master)
print("App name      :", spark.sparkContext.appName)
print("Default parallelism:", spark.sparkContext.defaultParallelism)

## Lab 1 — Kenali cluster yang sedang dipakai (15–20 menit)
**Sebelum menjalankan kode, prediksi:** berapa core total cluster dan di mana Driver berjalan?

Buka juga:
- Master UI: http://localhost:8080
- Worker UI: http://localhost:8081 dan http://localhost:8082
- Application UI: http://localhost:4040 (selama SparkSession aktif)

Catat jumlah worker, core, memory, dan executor yang terlihat.

**Rujukan slide:** Deck 01, slides 27–28.

In [ ]:
print("Application ID:", spark.sparkContext.applicationId)
print("Default parallelism:", spark.sparkContext.defaultParallelism)
print("UI URL dari SparkContext:", spark.sparkContext.uiWebUrl)

**Observasi peserta**
1. Apakah jumlah core cluster sama dengan jumlah core yang digunakan aplikasi?
2. Apakah Driver merupakan Worker?
3. Mengapa aplikasi kedua dapat menunggu resource walaupun cluster masih hidup?

## Lab 2 — Partition → Task (20 menit)
Buat DataFrame dengan 4 partition, lalu ubah menjadi 8. Amati `count()` pada Spark UI.

**Rujukan slide:** Deck 01, slides 29–30.

In [ ]:
df2 = spark.range(0, 1_000_000, 1, 2)

print("df2 partitions:", df2.rdd.getNumPartitions())

In [ ]:
print("Jumlah baris:", df2.count())

In [ ]:
df4 = spark.range(0, 1_000_000, 1, 4)

print("df4 partitions:", df4.rdd.getNumPartitions())

In [ ]:
print("Jumlah baris:", df4.count())

In [ ]:
df8 = df4.repartition(8)

print("df8 partitions:", df8.rdd.getNumPartitions())

In [ ]:
print("Jumlah baris:", df8.count())

In [ ]:
# Lihat jumlah record per partition.
partition_profile = (
    df8.rdd
    .mapPartitionsWithIndex(lambda idx, it: [(idx, sum(1 for _ in it))])
    .collect()
)
partition_profile

**Challenge:** Jika stage memiliki 8 task tetapi hanya 4 slot task tersedia, berapa gelombang minimal yang dibutuhkan bila durasi task seragam? Jelaskan hubungan partition, task, dan core.

## Lab 3.1 — Transformation, action (20 menit)
Jalankan cell pertama dan lihat Spark UI. Apakah job baru muncul? Baru setelah itu jalankan action.

**Rujukan slide:** Deck 01, slide 32.

In [ ]:
base = spark.range(0, 2_000_000, 1, 4)
transformed = (
    base
    .filter((F.col("id") % 2) == 0)
    .withColumn("bucket", F.col("id") % 10)
    .select("id", "bucket")
)
print("DataFrame sudah didefinisikan. Belum ada action eksplisit.")

In [ ]:
# ACTION: memicu komputasi
result_count = transformed.count()
print("Rows:", result_count)

In [ ]:
# ACTION: memicu komputasi
transformed.show()

**Pertanyaan:** `filter`, `withColumn`, `select`, `count` termasuk transformation atau action? Apa bukti yang terlihat pada UI?

## Lab 3.2 — Lazy evaluation

**Rujukan slide:** Deck 01, slides 40–41.

In [ ]:
base_df = spark.range(0, 10_000_000, 1, 4)

filtered = base_df.filter(
    F.col("id") % 2 == 0
)

projected = filtered.select("id")

In [ ]:
# Amati UI SEBELUM action ini:
answer = projected.count()
print("Hasil:", answer)

In [ ]:
from pyspark.sql import functions as F

df = spark.range(0, 10_000_000, 1, 4)

result = (
    df
    .withColumn("group_id", F.col("id") % 10)
    .groupBy("group_id")
    .count()
)

result.show()


## Lab 4 — Dari plan ke shuffle (25 menit)
`groupBy` adalah wide transformation. Cari `Exchange` pada physical plan dan cocokkan dengan stage boundary di UI.

**Rujukan slide:** Deck 01, slides 42–43.

In [ ]:
agg = (
    transformed
    .groupBy("bucket")
    .agg(F.count("*").alias("n"), F.sum("id").alias("sum_id"))
    .orderBy("bucket")
)
agg.explain("formatted")

In [ ]:
agg.show(truncate=False)

**Checklist Spark UI:**
- Job mana yang muncul karena `show()`?
- Berapa stage?
- Stage mana yang memiliki shuffle read/write?
- Berapa task pada masing-masing stage?

Jangan menganggap satu pemanggilan API = satu task.

## Lab 5 — Eksperimen jumlah partition (30 menit)
Ini **bukan benchmark ilmiah**. Tujuannya melihat trade-off parallelism vs scheduling overhead.

| Partisi | Waktu (detik) | Stage | Jumlah task | Pengamatan |
|---:|---:|---:|---:|---|
| 1 | | | | |
| 2 | | | | |
| 4 | | | | |
| 8 | | | | |

**Rujukan slide:** Deck 01, slides 44–45.

In [ ]:
rows = []
for n in [1, 2, 4, 8]:
    sample = spark.range(0, 10_000_000, 1, n)
    t0 = time.perf_counter()
    total = sample.agg(F.sum("id")).collect()[0][0]
    elapsed = time.perf_counter() - t0
    rows.append((n, round(elapsed, 3), total))

spark.createDataFrame(rows, ["partitions", "seconds", "sum_id"]).show(truncate=False)

**Expected checksum:** `49,999,995,000,000`.

Diskusikan:
- Mengapa 8 partition tidak selalu 8× lebih cepat daripada 1 partition?
- Apa yang terjadi jika jumlah task jauh lebih banyak daripada core?
- Mengapa pengukuran perlu diulang?

## Exit ticket
Tulis jawaban singkat:
1. Worker, Executor, Task, Partition: apa bedanya?
2. Apa yang memicu Job?
3. Mengapa `Exchange` penting saat membaca plan?
4. Mengapa `repartition()` bukan optimasi gratis?